# 11 kW OBC Charging Profile → Operating Points

This notebook models a **11 kW On-Board Charger (OBC)** charging a traction battery
from 10 % to 100 % State-of-Charge (SoC) using a CC-CV charging strategy and converts
the resulting voltage/current profile into a weighted histogram of operating points.

**Workflow**
1. Generate a dense SoC sweep from 0.1 to 1.0 (200 points).
2. Call `obc_profile` to compute battery voltage `V_bat` and charging current `I_bat`
   at each SoC step for an 11 kW charger.
3. Visualise the CC-CV profile.
4. Convert the profile to a 6-bin histogram for loss-map evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyplecs.mission_profile import obc_profile, mission_profile_to_histogram

In [ ]:
# Generate SoC sweep and compute OBC operating points at 11 kW
soc = np.linspace(0.1, 1.0, 200)
result = obc_profile(soc=soc, p_max_kw=11.0)

print(f"V_bat range : {result['V_bat'].min():.1f} – {result['V_bat'].max():.1f} V")
print(f"I_bat range : {result['I_bat'].min():.1f} – {result['I_bat'].max():.1f} A")
print(f"P_obc range : {result['P_obc'].min():.2f} – {result['P_obc'].max():.2f} kW")

In [ ]:
# Plot V_bat and I_bat vs SoC
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax1.plot(soc * 100, result["V_bat"], color="tab:blue", linewidth=2)
ax1.set_ylabel("Battery Voltage (V)")
ax1.set_title("11 kW OBC — CC-CV Charging Profile")
ax1.grid(True, alpha=0.4)

ax2.plot(soc * 100, result["I_bat"], color="tab:orange", linewidth=2)
ax2.set_xlabel("State of Charge (%)")
ax2.set_ylabel("Charging Current (A)")
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# Build operating-point DataFrame and convert to histogram
ops = pd.DataFrame({
    "V": result["V_bat"],
    "I": result["I_bat"],
})

tbl = mission_profile_to_histogram(ops, columns=["V", "I"], n_bins=6)
tbl

In [ ]:
print(tbl.summary())

fig, ax = plt.subplots(figsize=(8, 4))
tbl.plot(ax=ax, kind="bar")
ax.set_title("11 kW OBC — Operating-Point Histogram (6 bins)")
plt.tight_layout()
plt.show()